# SU(4) hybrid-contraction adapter audit
Run the single code cell. Upload the full symbolic source bundle, the SU(4) enumerator bundle, and the SU(4) local-algebra bundle when prompted.

In [ ]:
#!/usr/bin/env python3
"""
SU(4) hybrid-contraction adapter audit.

Purpose
-------
Freeze the exact interface between the verified stable-rank local library and
the global partition-loop contraction before inserting the SU(4) epsilon /
delta-epsilon projectors.

Inputs
------
1. Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE*.zip
2. SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip
3. SU4_LOCAL_ALGEBRA_V1_BUNDLE.zip

The script:
  * locates and imports y4_sun_walled_brauer_fixed_rank.py;
  * extracts full AST source bodies for LocalLibrary, contract_choice,
    merge_partitions, min_fill, build_corpus, folded_coeff, and extract_qab;
  * loads all balanced signatures that occur beside SU(4) exceptional links;
  * records the exact Python type/shape/schema of every LocalLibrary.get term;
  * verifies LocalLibrary(4) works on the required balanced signatures;
  * constructs representative balanced-only contract_choice calls;
  * verifies the exported SU(4) local library has all 42 signatures and 78
    joint channels;
  * emits a machine-readable adapter contract for the next patch.

No physics result is inferred here. This stage prevents an unsafe guess about
the compact stable term representation.

Outputs
-------
SU4_HYBRID_ADAPTER_AUDIT_V1/SU4_HYBRID_ADAPTER_AUDIT_V1.json
SU4_HYBRID_ADAPTER_AUDIT_V1/SU4_HYBRID_ADAPTER_AUDIT_V1.md
SU4_HYBRID_ADAPTER_AUDIT_V1/source_contract_snippets.py
SU4_HYBRID_ADAPTER_AUDIT_V1_BUNDLE.zip
"""
from __future__ import annotations

import ast
import gzip
import hashlib
import importlib.util
import inspect
import json
import os
import re
import sys
import tempfile
import time
import zipfile
from collections import Counter, defaultdict
from fractions import Fraction
from pathlib import Path
from typing import Any, Iterable

VERSION = "2026-06-14-su4-hybrid-adapter-audit-v1"
BASE = Path("/content") if Path("/content").exists() else Path("/mnt/data")
OUT = BASE / "SU4_HYBRID_ADAPTER_AUDIT_V1"
EXTRACT = OUT / "extracted"
OUT.mkdir(parents=True, exist_ok=True)
EXTRACT.mkdir(parents=True, exist_ok=True)

SOURCE_GLOB = "Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE*.zip"
ENUM_GLOB = "SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE*.zip"
ALG_GLOB = "SU4_LOCAL_ALGEBRA_V1_BUNDLE*.zip"

FIXED_NAME = "y4_sun_walled_brauer_fixed_rank.py"
EXC_NAME = "y4_su4_exceptional_only_words.json.gz"
LOCAL_LIB_NAME = "y4_su4_exceptional_local_library.json"


def gate(name: str, cond: bool, detail: str = "") -> None:
    status = "PASS" if cond else "FAIL"
    print(f"{status:4s} {name:90s} {detail}")
    if not cond:
        raise AssertionError(f"{name}: {detail}")


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def safe_extract(zpath: Path, dest: Path) -> list[Path]:
    dest.mkdir(parents=True, exist_ok=True)
    root = dest.resolve()
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            target = (dest / info.filename).resolve()
            if target != root and not str(target).startswith(str(root) + os.sep):
                raise ValueError(f"unsafe ZIP member: {info.filename}")
        zf.extractall(dest)
        return [dest / i.filename for i in zf.infolist() if not i.is_dir()]


def recursive_extract(archives: Iterable[Path], max_depth: int = 5) -> list[dict[str, Any]]:
    queue = [(Path(p), 0) for p in archives]
    seen: set[str] = set()
    records: list[dict[str, Any]] = []
    while queue:
        zp, depth = queue.pop(0)
        if depth > max_depth or not zp.is_file():
            continue
        h = sha256(zp)
        if h in seen:
            continue
        seen.add(h)
        label = re.sub(r"[^A-Za-z0-9_.-]+", "_", zp.stem)[:100]
        dest = EXTRACT / f"d{depth}_{label}_{h[:10]}"
        try:
            files = safe_extract(zp, dest)
            records.append({
                "archive": str(zp), "sha256": h, "depth": depth,
                "destination": str(dest), "file_count": len(files), "status": "ok",
            })
            for p in files:
                if p.suffix.lower() == ".zip":
                    queue.append((p, depth + 1))
        except Exception as exc:
            records.append({
                "archive": str(zp), "sha256": h, "depth": depth,
                "status": "error", "error": repr(exc),
            })
    return records


def find_all(name: str, roots: Iterable[Path]) -> list[Path]:
    found: dict[str, Path] = {}
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(name):
                if p.is_file():
                    found[sha256(p)] = p
        except Exception:
            pass
    return sorted(found.values(), key=lambda p: str(p))


def find_glob(pattern: str, roots: Iterable[Path]) -> list[Path]:
    found: dict[str, Path] = {}
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in root.rglob(pattern):
                if p.is_file():
                    found[sha256(p)] = p
        except Exception:
            pass
    return sorted(found.values(), key=lambda p: str(p))


def upload_if_needed() -> list[Path]:
    roots = [BASE]
    have_source = bool(find_glob(SOURCE_GLOB, roots) or find_all(FIXED_NAME, roots))
    have_enum = bool(find_glob(ENUM_GLOB, roots) or find_all(EXC_NAME, roots))
    have_alg = bool(find_glob(ALG_GLOB, roots) or find_all(LOCAL_LIB_NAME, roots))
    if have_source and have_enum and have_alg:
        return []
    if not Path("/content").exists():
        return []
    try:
        from google.colab import files as colab_files  # type: ignore
    except Exception:
        return []

    print("\nUPLOAD REQUIRED — select all missing files:")
    if not have_source:
        print("  Y4_SUN_WALLED_BRAUER_FULL_SYMBOLIC_BUNDLE_2026-06-14_V2*.zip")
    if not have_enum:
        print("  SU4_EXCEPTIONAL_ENUMERATOR_V1_BUNDLE.zip")
    if not have_alg:
        print("  SU4_LOCAL_ALGEBRA_V1_BUNDLE.zip")
    uploaded = colab_files.upload()
    saved: list[Path] = []
    for name, data in uploaded.items():
        target = Path("/content") / Path(name).name
        target.write_bytes(data)
        saved.append(target)
        print(f"saved {len(data):,} bytes -> {target}")
    return saved


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def read_json_gz(path: Path) -> Any:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return json.load(f)


def jsonable(value: Any, depth: int = 0) -> Any:
    if depth > 7:
        return "<max-depth>"
    if value is None or isinstance(value, (bool, int, float, str)):
        return value
    if isinstance(value, Fraction):
        return {"type": "Fraction", "value": str(value)}
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, tuple):
        return {
            "type": "tuple",
            "length": len(value),
            "items": [jsonable(x, depth + 1) for x in value],
        }
    if isinstance(value, list):
        return {
            "type": "list",
            "length": len(value),
            "items": [jsonable(x, depth + 1) for x in value],
        }
    if isinstance(value, dict):
        return {
            "type": "dict",
            "length": len(value),
            "items": [
                [jsonable(k, depth + 1), jsonable(v, depth + 1)]
                for k, v in list(value.items())[:40]
            ],
        }
    if hasattr(value, "tolist"):
        try:
            return {
                "type": type(value).__name__,
                "tolist": jsonable(value.tolist(), depth + 1),
            }
        except Exception:
            pass
    return {
        "type": f"{type(value).__module__}.{type(value).__qualname__}",
        "repr": repr(value)[:4000],
    }


def structural_signature(value: Any, depth: int = 0) -> Any:
    if depth > 6:
        return "..."
    if isinstance(value, Fraction):
        return "Fraction"
    if value is None:
        return "None"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, int):
        return "int"
    if isinstance(value, str):
        return "str"
    if isinstance(value, tuple):
        return ("tuple", len(value), tuple(structural_signature(x, depth + 1) for x in value))
    if isinstance(value, list):
        return ("list", len(value), tuple(structural_signature(x, depth + 1) for x in value))
    if isinstance(value, dict):
        return (
            "dict",
            tuple(sorted(
                (repr(k), structural_signature(v, depth + 1))
                for k, v in value.items()
            )),
        )
    return f"{type(value).__module__}.{type(value).__qualname__}"


def extract_ast_bodies(source_path: Path, names: list[str]) -> dict[str, str]:
    source = source_path.read_text(encoding="utf-8")
    lines = source.splitlines()
    tree = ast.parse(source)
    out: dict[str, str] = {}

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if node.name in names and hasattr(node, "end_lineno"):
                text = "\n".join(lines[node.lineno - 1: node.end_lineno])
                key = node.name
                if key in out:
                    key = f"{key}@{node.lineno}"
                out[key] = text

    # Also extract selected methods from LocalLibrary separately.
    for node in tree.body:
        if isinstance(node, ast.ClassDef) and node.name == "LocalLibrary":
            for child in node.body:
                if isinstance(child, ast.FunctionDef) and child.name in {
                    "__init__", "perms", "gram", "c2", "casimir", "get"
                }:
                    if hasattr(child, "end_lineno"):
                        out[f"LocalLibrary.{child.name}"] = "\n".join(
                            lines[child.lineno - 1: child.end_lineno]
                        )
    return out


def collect_required_signatures(exc_payload: dict[str, Any]) -> tuple[set[tuple[int, ...]], set[tuple[int, ...]]]:
    balanced: set[tuple[int, ...]] = set()
    exceptional: set[tuple[int, ...]] = set()
    for word in exc_payload["words"]:
        for assignment in word["assignments"]:
            for local in assignment["local_records"]:
                exceptional.add(tuple(int(x) for x in local["tokens"]))
        # Reconstructing every balanced neighboring link requires the source
        # geometry and is done below. This function only collects the explicit
        # exceptional signatures from the manifest.
    return balanced, exceptional


def main() -> None:
    started = time.time()
    print("=" * 116)
    print("SU(4) HYBRID CONTRACTION ADAPTER AUDIT")
    print("=" * 116)
    print("version :", VERSION)
    print("output  :", OUT)

    uploaded = upload_if_needed()
    archives = (
        find_glob(SOURCE_GLOB, [BASE])
        + find_glob(ENUM_GLOB, [BASE])
        + find_glob(ALG_GLOB, [BASE])
        + [p for p in uploaded if p.suffix.lower() == ".zip"]
    )
    extraction = recursive_extract(archives)
    roots = [BASE, EXTRACT]

    fixed_paths = find_all(FIXED_NAME, roots)
    exc_paths = find_all(EXC_NAME, roots)
    alg_paths = find_all(LOCAL_LIB_NAME, roots)

    gate("fixed-rank contraction source located", bool(fixed_paths), str(fixed_paths[:2]))
    gate("SU(4) exceptional manifest located", bool(exc_paths), str(exc_paths[:2]))
    gate("SU(4) exceptional local library located", bool(alg_paths), str(alg_paths[:2]))

    fixed_path = fixed_paths[0]
    exc_path = exc_paths[0]
    alg_path = alg_paths[0]

    wb = load_module("y4_sun_walled_brauer_su4_adapter", fixed_path)
    required_symbols = [
        "LocalLibrary", "contract_choice", "merge_partitions", "min_fill",
        "build_corpus", "folded_coeff", "extract_qab", "pb", "rep",
    ]
    missing = [name for name in required_symbols if not hasattr(wb, name)]
    gate("fixed-rank engine exposes required adapter symbols", not missing, str(missing))

    exc_payload = read_json_gz(exc_path)
    alg_payload = json.loads(alg_path.read_text(encoding="utf-8"))

    gate("exceptional manifest contains 76 words",
         len(exc_payload["words"]) == 76, str(len(exc_payload["words"])))
    gate("exceptional manifest contains 312 assignments",
         int(exc_payload["counts"]["exceptional_assignments"]) == 312,
         str(exc_payload["counts"]["exceptional_assignments"]))
    gate("local algebra contains 42 signatures",
         len(alg_payload["signatures"]) == 42,
         str(len(alg_payload["signatures"])))
    gate("local algebra contains 78 joint channels",
         int(alg_payload["counts"]["joint_channels"]) == 78,
         str(alg_payload["counts"]["joint_channels"]))

    # Rebuild every local signature, not only the exceptional links listed in
    # local_records, so the adapter knows exactly which stable terms coexist.
    balanced_signatures: set[tuple[int, ...]] = set()
    exceptional_signatures: set[tuple[int, ...]] = set()
    representative_specs: list[dict[str, Any]] = []

    for word_record in exc_payload["words"]:
        factors = (
            [tuple(int(x) for x in word_record["root"])]
            + [tuple(int(x) for x in p) for p in word_record["ordered_insertions"]]
            + [tuple(int(x) for x in word_record["output"])]
        )
        for assignment in word_record["assignments"]:
            signs0 = tuple(int(x) for x in assignment["signs"])
            signs = tuple(wb.rep(signs0))
            eff = signs[:5] + (-signs[5],)
            links: dict[Any, list[tuple[int, int, int, int, int]]] = defaultdict(list)
            for event_index, plaquette in enumerate(factors):
                for edge, (link, incidence, start_corner, end_corner) in enumerate(wb.pb(plaquette)):
                    token = eff[event_index] * incidence
                    row_variable = 4 * event_index + (
                        start_corner if incidence == 1 else end_corner
                    )
                    col_variable = 4 * event_index + (
                        end_corner if incidence == 1 else start_corner
                    )
                    links[link].append(
                        (event_index, edge, token, row_variable, col_variable)
                    )

            specs = []
            for link, occurrences0 in sorted(links.items()):
                occurrences = tuple(sorted(occurrences0))
                signature = [0] * 6
                for event_index, edge, token, rv, cv in occurrences:
                    signature[event_index] = token
                signature = tuple(signature)
                rows = tuple(x[3] for x in occurrences)
                cols = tuple(x[4] for x in occurrences)
                specs.append((signature, rows, cols))
                charge = signature.count(1) - signature.count(-1)
                if charge == 0:
                    balanced_signatures.add(signature)
                else:
                    exceptional_signatures.add(signature)

            if len(representative_specs) < 20:
                representative_specs.append({
                    "word_id": word_record["su4_exceptional_id"],
                    "stable_ordered_id": word_record.get("stable_ordered_id"),
                    "signs": list(signs),
                    "specs": [
                        {
                            "signature": list(sig),
                            "rows": list(rows),
                            "cols": list(cols),
                        }
                        for sig, rows, cols in specs
                    ],
                })

    gate("reconstructed exceptional signature set matches local algebra",
         exceptional_signatures
         == {tuple(int(x) for x in row["signature"]) for row in alg_payload["signatures"]},
         f"reconstructed={len(exceptional_signatures)} library={len(alg_payload['signatures'])}")
    gate("balanced signatures coexist in exceptional assignments",
         bool(balanced_signatures), str(len(balanced_signatures)))

    lib = wb.LocalLibrary(4)
    term_schema_hist = Counter()
    term_count_hist = Counter()
    term_examples: dict[str, Any] = {}
    all_terms_json: dict[str, Any] = {}
    failures = []

    for signature in sorted(balanced_signatures):
        try:
            terms = lib.get(signature)
        except Exception as exc:
            failures.append({"signature": signature, "error": repr(exc)})
            continue
        gate("LocalLibrary.get returns a nonempty term list at N=4",
             bool(terms), f"signature={signature}")
        term_count_hist[len(terms)] += 1
        schemas = [repr(structural_signature(term)) for term in terms]
        for schema in schemas:
            term_schema_hist[schema] += 1
            if schema not in term_examples:
                term_examples[schema] = jsonable(terms[schemas.index(schema)])
        all_terms_json[repr(signature)] = {
            "term_count": len(terms),
            "schemas": schemas,
            "first_terms": [jsonable(term) for term in terms[:4]],
        }

    gate("all required balanced signatures build at N=4",
         not failures, str(failures[:5]))
    gate("stable local library produced at least one compact term schema",
         bool(term_schema_hist), str(len(term_schema_hist)))

    # Source extraction is authoritative for the next patch.
    source_names = [
        "LocalLibrary", "contract_choice", "merge_partitions", "min_fill",
        "build_corpus", "folded_coeff", "extract_qab"
    ]
    source_bodies = extract_ast_bodies(fixed_path, source_names)
    for name in source_names:
        gate(f"AST source body extracted: {name}",
             name in source_bodies or any(k.startswith(name + "@") for k in source_bodies),
             "")

    snippets_path = OUT / "source_contract_snippets.py"
    snippets_text = "\n\n# " + "\n\n# ".join(
        f"{name}\n{body}" for name, body in sorted(source_bodies.items())
    )
    snippets_path.write_text(snippets_text, encoding="utf-8")

    # Check contract_choice signature and source-level dependency names.
    contract_signature = str(inspect.signature(wb.contract_choice))
    get_signature = str(inspect.signature(wb.LocalLibrary.get))
    contract_source = inspect.getsource(wb.contract_choice)
    dependency_tokens = sorted(set(re.findall(
        r"\b(?:lib|get|merge_partitions|Fraction|rows|cols|energy|order|choice|term)\w*\b",
        contract_source
    )))

    gate("contract_choice has the expected five-argument interface",
         len(inspect.signature(wb.contract_choice).parameters) == 4,
         contract_signature)
    # Note: source currently shows contract_choice(specs, choices, lib, order):
    # four parameters. The message says five-argument semantically only if self
    # were present; enforce actual reflection count.
    gate("LocalLibrary.get has the expected self+signature interface",
         len(inspect.signature(wb.LocalLibrary.get).parameters) == 2,
         get_signature)

    # Verify that representative balanced-only subsets can be passed through
    # the unchanged contractor. We choose any reconstructed topology whose
    # links are all balanced; if none exists among exceptional assignments,
    # run a one-link synthetic adapter smoke test only at the schema level.
    balanced_contract_samples = []
    for record in representative_specs:
        specs = [
            (
                tuple(item["signature"]),
                tuple(item["rows"]),
                tuple(item["cols"]),
            )
            for item in record["specs"]
        ]
        if specs and all(sig.count(1) == sig.count(-1) for sig, _, _ in specs):
            scopes = []
            for sig, rows, cols in specs:
                scopes.extend((rows, cols))
            order, width = wb.min_fill(scopes)
            choices = tuple(0 for _ in specs)
            raw, energy = wb.contract_choice(specs, choices, lib, order)
            balanced_contract_samples.append({
                "word_id": record["word_id"],
                "raw": str(raw),
                "energy": jsonable(energy),
                "order": list(order),
                "width": width,
            })
            break

    summary = {
        "version": VERSION,
        "status": "PASS",
        "inputs": {
            "fixed_source": str(fixed_path),
            "fixed_source_sha256": sha256(fixed_path),
            "exceptional_manifest": str(exc_path),
            "exceptional_manifest_sha256": sha256(exc_path),
            "local_algebra": str(alg_path),
            "local_algebra_sha256": sha256(alg_path),
            "extraction": extraction,
        },
        "engine_interface": {
            "contract_choice_signature": contract_signature,
            "local_get_signature": get_signature,
            "contract_choice_dependency_tokens": dependency_tokens,
            "source_bodies": sorted(source_bodies),
            "source_snippets": str(snippets_path),
            "source_snippets_sha256": sha256(snippets_path),
        },
        "signature_sets": {
            "balanced_required": len(balanced_signatures),
            "exceptional_required": len(exceptional_signatures),
            "balanced_signatures": [list(x) for x in sorted(balanced_signatures)],
            "exceptional_signatures": [list(x) for x in sorted(exceptional_signatures)],
        },
        "stable_term_schema": {
            "schema_count": len(term_schema_hist),
            "schema_histogram": dict(term_schema_hist),
            "term_count_histogram": {
                str(k): v for k, v in sorted(term_count_hist.items())
            },
            "examples": term_examples,
            "terms_by_signature": all_terms_json,
        },
        "representative_specs": representative_specs,
        "balanced_contract_samples": balanced_contract_samples,
        "local_algebra_counts": alg_payload["counts"],
        "next_stage": (
            "Implement a HybridLocalLibrary whose get(signature) returns the exact stable "
            "term schema for balanced signatures and translated epsilon/delta-epsilon channel "
            "terms for the 42 exceptional signatures. Reuse contract_choice, folded_coeff, "
            "min_fill, and extract_qab without modification."
        ),
        "elapsed_seconds": time.time() - started,
    }

    json_path = OUT / "SU4_HYBRID_ADAPTER_AUDIT_V1.json"
    json_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")

    md = f"""# SU(4) hybrid-contraction adapter audit

**Status:** PASS  
**Version:** `{VERSION}`

## Exact interface recovered

```text
contract_choice{contract_signature}
LocalLibrary.get{get_signature}
```

Required local signatures:

```text
balanced signatures:    {len(balanced_signatures)}
exceptional signatures: {len(exceptional_signatures)}
stable term schemas:    {len(term_schema_hist)}
```

The stable local library successfully generated every balanced signature needed
by the 312 exceptional assignments at `N=4`.

The full source bodies of `LocalLibrary`, `contract_choice`,
`merge_partitions`, `min_fill`, `build_corpus`, `folded_coeff`, and
`extract_qab` are preserved in `{snippets_path.name}`.

## Next stage

Construct `HybridLocalLibrary` by translating each exported SU(4) channel

```text
sum_ab K_ab T_a(row) T_b(col)
```

into the recovered stable compact-term schema. Then reuse the existing exact
partition-loop contraction and `extract_qab` unchanged.
"""
    md_path = OUT / "SU4_HYBRID_ADAPTER_AUDIT_V1.md"
    md_path.write_text(md, encoding="utf-8")

    bundle = BASE / "SU4_HYBRID_ADAPTER_AUDIT_V1_BUNDLE.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in (json_path, md_path, snippets_path):
            zf.write(p, arcname=p.name)
        source_path = Path(globals().get("__file__", ""))
        if source_path.is_file():
            zf.write(source_path, arcname=source_path.name)

    print("\n" + "=" * 116)
    print("SU(4) HYBRID ADAPTER AUDIT STATUS: PASS")
    print("=" * 116)
    print("balanced signatures required :", len(balanced_signatures))
    print("exceptional signatures       :", len(exceptional_signatures))
    print("stable compact term schemas  :", len(term_schema_hist))
    print("term-count histogram         :", dict(sorted(term_count_hist.items())))
    print("contract_choice signature    :", contract_signature)
    print("LocalLibrary.get signature   :", get_signature)
    print("JSON:", json_path, sha256(json_path))
    print("MD:  ", md_path, sha256(md_path))
    print("SRC: ", snippets_path, sha256(snippets_path))
    print("ZIP: ", bundle, sha256(bundle))
    print("=" * 116)


if __name__ == "__main__":
    main()
